환경/라이브러리

In [1]:
import os
import pandas as pd
import numpy as np

경로/설정

In [2]:
PROJECT_ROOT = os.path.dirname(os.getcwd())

RAW_PATH = os.path.join(PROJECT_ROOT, "data", "raw", "statcast_2025.csv")

PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

OUT_PATH = os.path.join(PROCESSED_DIR, "pitch_clean.parquet")

print("NOTEBOOK CWD:", os.getcwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_PATH:", RAW_PATH)
print("OUT_PATH:", OUT_PATH)
print("RAW exists?:", os.path.exists(RAW_PATH))

NOTEBOOK CWD: c:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\notebooks
PROJECT_ROOT: c:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess
RAW_PATH: c:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\data\raw\statcast_2025.csv
OUT_PATH: c:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\data\processed\pitch_clean.parquet
RAW exists?: True


데이터 로드

In [3]:
df = pd.read_csv(RAW_PATH, low_memory=False)
print("raw shape:", df.shape)
df.head(3)

raw shape: (770795, 118)


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,NaN,2025-03-20,NaN,NaN,NaN,"Ventura, Jordany",670869,683094,strikeout,swinging_strike,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2025-03-20,NaN,NaN,NaN,"Ventura, Jordany",670869,683094,NaN,foul,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2025-03-20,NaN,NaN,NaN,"Ventura, Jordany",670869,683094,NaN,foul,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


컬럼 선택

In [4]:
selected_cols = [
    'pitch_type', 'game_date', 'release_speed', 'release_pos_x', 'release_pos_z',
    'batter', 'pitcher', 'events', 'description', 'zone', 'stand', 'p_throws',
    'bb_type', 'balls', 'strikes', 'pfx_x', 'pfx_z', 'plate_x', 'plate_z',
    'on_3b', 'on_2b', 'on_1b', 'outs_when_up', 'inning', 'inning_topbot',
    'launch_speed', 'launch_angle', 'release_spin_rate', 'release_extension',
    'pitch_number', 'bat_score_diff', 'arm_angle'
]

missing = [c for c in selected_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns in raw data: {missing}")

df = df[selected_cols].copy()
print("after select shape:", df.shape)

after select shape: (770795, 32)


타입 정리

In [5]:
df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")

# id 컬럼
df["pitcher"] = pd.to_numeric(df["pitcher"], errors="coerce")
df["batter"] = pd.to_numeric(df["batter"], errors="coerce")

# count
df["balls"] = pd.to_numeric(df["balls"], errors="coerce").fillna(0).astype(int)
df["strikes"] = pd.to_numeric(df["strikes"], errors="coerce").fillna(0).astype(int)

# 기타 숫자형 (이후 UMAP 입력 안정화)
num_cols = [
    "release_speed","release_spin_rate","pfx_x","pfx_z",
    "release_pos_x","release_pos_z","release_extension","arm_angle",
    "plate_x","plate_z","launch_speed","launch_angle","bat_score_diff",
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

결측치 처리

In [6]:
# 인플레이 아닐 때 NaN -> 0
df["launch_speed"] = df["launch_speed"].fillna(0.0)
df["launch_angle"] = df["launch_angle"].fillna(0.0)

# 주자 유무: 있으면 1, 없으면 0
df["on_1b"] = df["on_1b"].notna().astype(int)
df["on_2b"] = df["on_2b"].notna().astype(int)
df["on_3b"] = df["on_3b"].notna().astype(int)

description 그룹화 컬럼 생성

In [7]:
DESC_TO_GROUP = {
    "called_strike": "strike",
    "swinging_strike": "strike",
    "swinging_strike_blocked": "strike",
    "missed_bunt": "strike",

    "foul": "foul",
    "foul_tip": "foul",
    "foul_bunt": "foul",

    "ball": "ball",
    "blocked_ball": "ball",
    "pitchout": "ball",
    "intent_ball": "ball",

    "hit_into_play": "inplay",
    "hit_into_play_score": "inplay",
    "hit_into_play_no_out": "inplay",
}

df["description_group"] = df["description"].map(DESC_TO_GROUP).fillna("other")
df["description_group"].value_counts().head(10)

description_group
ball      274423
strike    210261
foul      146569
inplay    134759
other       4783
Name: count, dtype: int64

events 그룹화 컬럼 생성

In [8]:
EVENT_TO_GROUP = {
    "single": "hit",
    "double": "hit",
    "triple": "hit",
    "home_run": "hit",

    "walk": "walk",
    "intent_walk": "walk",
    "hit_by_pitch": "walk",

    "strikeout": "out",
    "strikeout_double_play": "out",
    "field_out": "out",
    "force_out": "out",
    "double_play": "out",
    "grounded_into_double_play": "out",

    "field_error": "reached",
}

df["events_group"] = df["events"].map(EVENT_TO_GROUP).fillna("other")
df["events_group"].value_counts().head(10)

events_group
other      575804
out        131580
hit         43440
walk        18859
reached      1112
Name: count, dtype: int64

UMAP 입력 feature 정의 + 필수 결측 제거

In [9]:
PITCH_FEATURES_FOR_UMAP = [
    "release_speed",
    "release_spin_rate",
    "pfx_x",
    "pfx_z",
    "release_pos_x",
    "release_pos_z",
    "release_extension",
    "arm_angle",
]

# pitcher/batter 없는 행 제거
df = df.dropna(subset=["pitcher", "batter"])

# UMAP feature NaN 제거
before = len(df)
df = df.dropna(subset=PITCH_FEATURES_FOR_UMAP)
after = len(df)
print(f"dropna UMAP features: {before:,} -> {after:,}")

dropna UMAP features: 770,795 -> 721,038


파생 변수: base_state, count

In [10]:
df["base_state"] = df["on_1b"] + df["on_2b"] * 2 + df["on_3b"] * 4
df["count"] = df["balls"].astype(str) + "-" + df["strikes"].astype(str)

저장 (parquet)

In [11]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


c:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\.venv\Scripts\python.exe: No module named pip


In [12]:
df.to_parquet(OUT_PATH, index=False)
print("saved:", OUT_PATH)
print("final shape:", df.shape)
df.head(3)

saved: c:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\data\processed\pitch_clean.parquet
final shape: (721038, 36)


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,batter,pitcher,events,description,zone,...,launch_angle,release_spin_rate,release_extension,pitch_number,bat_score_diff,arm_angle,description_group,events_group,base_state,count
6750,FF,2025-03-19,92.8,0.11,5.98,807713,681911,field_out,hit_into_play,3.0,...,-18.0,2526.0,6.4,2,-3,71.0,inplay,out,3,0-1
6751,SL,2025-03-19,83.5,0.31,6.18,807713,681911,NaN,called_strike,1.0,...,0.0,2465.0,6.3,1,-3,66.9,strike,other,3,0-0
6752,FF,2025-03-19,92.6,0.22,6.00,457759,681911,walk,ball,12.0,...,0.0,2532.0,6.4,5,-3,62.3,ball,walk,2,3-1


투수별 샘플 수 체크

In [13]:
pitch_counts = df.groupby("pitcher").size().sort_values(ascending=False)
print(pitch_counts.describe())
pitch_counts.head(20)

count     872.000000
mean      826.878440
std       814.781114
min         2.000000
25%       180.500000
50%       605.500000
75%      1124.250000
max      3466.000000
dtype: float64


pitcher
592332    3466
607074    3363
642547    3362
808967    3315
657277    3282
608331    3262
676979    3241
669373    3152
668678    3134
666200    3120
656302    3112
579328    3094
650911    3082
592662    3069
676440    3051
668909    3003
694973    2997
622491    2976
686613    2962
605135    2947
dtype: int64